# Post-quantum IPsec, end to end
## Containers, router CLI, Ansible

How do you add post-quantum protection to an IPsec tunnel, and what does it cost?
This notebook answers that in three acts, each one closer to production: strongSwan
in containers, then three IOS XE 26.2 routers at the CLI, then the same routers with
Ansible over NETCONF. Every step says what to run, what you should see, and why.

**First time?** Build your lab with [`demos/README.md`](../README.md) (prerequisites,
underlay, credentials). This notebook assumes that's done, and checks it for you in
[Before each run](#preflight).

Three kinds of cell:

- **Act 1 cells** drive the strongSwan containers and draw the handshake as it
  happens.
- **Router checks** (`r1(...)`, `router_check(...)`) run read-only `show`
  commands through Ansible and draw expected vs measured. They never send
  configuration: in Act 2 you type every change yourself, and the checks confirm
  you got the result. The one exception is `setup()`, which configures the
  routers and says so.
- **Playbook cells** (`playbook(...)`) run Act 3 from
  `deploy/c8000/automation` with its own venv, keep the `PLAY RECAP` visible
  and draw a per-router panel.

## Contents

- [Setup](#setup)
- [Before each run](#preflight): containers, underlay, routers, automation
- [Act 1: containers](#act1): Demo 1 and Demo 2
- [Starting at Act 2?](#act2-setup): set up the routers without the containers
- [Act 2: live IOS XE CLI](#act2): Demos 3 to 7
- [Act 3: Ansible](#act3): Demo 8 and Demo 9
- [Clean up](#after)
- [Run it again](#again)

<a id="setup"></a>
## Setup

Run this once per kernel. It finds `pqc_demo.py` in `demos/containers_demo/`, the
shared helpers in `demos/common/` and `ipsec_lab.py` next to this notebook,
whichever folder Jupyter started in.

In [ ]:
import sys, pathlib, re
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'ipsec_demo'):
    sys.path.insert(0, str(_demos / _p))
from pqc_demo import *        # noqa: F401,F403  containers, panels
from ipsec_lab import *  # noqa: F401,F403  routers, playbooks, labs

# Ladders for the Act 1 panels (same objects the learner notebook uses).
ipsec = IPsecDemo(); ipsec_auth = IPsecAuthDemo()

set_presenter_mode(True)
print('Repo:       ', REPO_ROOT)
print('Automation: ', AUTOMATION_DIR)
print('Containers:  initiator responder | auth_initiator auth_responder | host')
print('Routers:     r1 r2 r3 (read-only)   Playbooks: playbook(), playbook_bg(), playbook_wait()')

---
<a id="preflight"></a>
## Before each run

Most failures are leftovers from the last run, so check everything before you
start. If a check fails, the fix is in the table below, in
[Build your lab](../README.md#build-your-lab), or in this demo's
[`README.md`](README.md).

### Containers

**First time on this machine only:** build this demo's labs. It's slow the first time, because the experimental strongSwan `master` build is compiled from source with `--no-cache`.

In [ ]:
# host: build the Act 1 images (first time on this machine)
labs.build()

In [ ]:
# host: bring this demo's stacks up, then show the readiness board
labs.up()

If anything in Act 1 sticks later, `labs.panic()` stops stray daemons and
captures without taking the stacks down.

The board below should be all green. No Docker? The container rows fail, and
that's fine if you start at Act 2.

In [ ]:
# host: Act 1 containers up, Act 2/3 prerequisites in place
labs.status()

<a id="underlay"></a>
### Your lab, as the notebook sees it

Every router cell reads interfaces and addresses from `demos/common/lab.yml`,
or the repo's `topology.yml` for anything it leaves out. Is this what you
built? If not, see [Build your lab](../README.md#build-your-lab).

In [ ]:
# laptop: underlay and MACsec link, as the router cells see them
underlay()

### Routers: is the lab in Act 2's starting state?

The target is simple: **R1 and R3 run the Exercise 1 classical tunnel and
nothing post-quantum, R2 is plain transit, and there are no automation
objects anywhere.** The ML-DSA trustpoint is the one exception: it's
imported on R1 and R3 in advance, because Demo 7 needs it.

This cell checks all of it, read-only, in parallel (about 15 seconds). Starting
from scratch? [`setup()`](#act2-setup) fixes every row except the underlay.

In [ ]:
# R1, R2, R3: read-only checks of the Act 2 starting state
# (the full list is check() in ipsec_lab.py; addresses and interfaces come from underlay())
check()

**If a row fails**, here's where the fix lives:

| Row | Fix |
|---|---|
| underlay address, underlay ping | [Build your lab](../README.md#build-your-lab) |
| Exercise 1 proposal, Tunnel0, PSK SA | [`setup()`](#act2-setup) pastes it with your underlay. By hand: Exercise 1 from `deploy/c8000/ipsec.md` on R1 and R3 (the PSK lives there, not here), with `tunnel source`, the keyring peer address, the identity match and `tunnel destination` changed to what `underlay()` shows |
| PQC present | `no pqc mlkem768 optional` under the proposal, `no crypto ikev2 fragmentation` |
| automation objects | the teardown cells in [Clean up](#after) |
| ML-DSA profile present | the reset in [Act 3](#act3), "Get the routers ready" |
| trustpoint missing | [`setup()`](#act2-setup), or [by hand](README.md#ml-dsa-trustpoints-by-hand) |
| leftover Demo 6 key | `crypto key zeroize mldsa PQC-DEMO` |
| service internal | `service internal` in config mode. Some early 26.2 builds need it before they accept ML-DSA certificates; on a release image you can drop this row |

Want to see the raw output behind a row? Ask the router directly:

In [ ]:
# R1: any single read-only command, real output
r1('show crypto ikev2 sa');

### Automation

`verify.yml` only reads (NETCONF and `show`), so it's safe at any time. It proves
the playbooks can reach all three routers and prints their crypto posture.

In [ ]:
# Ansible: read-only posture report across r1, r2, r3
playbook('verify.yml');

**`Could not open socket to <router>:830`?** NETCONF isn't on. `bootstrap.yml`
turns it on (that's a config change, so it's not part of the read-only checks)
and waits for port 830 to answer. Run it, then rerun `verify.yml`.

In [ ]:
# Ansible: enable NETCONF on r1, r2, r3 (only if verify.yml can't reach port 830)
playbook('ansible-playbook bootstrap.yml');

---
<a id="act1"></a>
## Act 1: learn it in containers

Two strongSwan stacks, an initiator and a responder, on one Docker network. One
cell per action: read the comment, run it, read the output.

### Demo 1: hybrid key exchange

What does adding ML-KEM to IKEv2 actually change? Look for:

- `CURVE_25519/KE1_ML_KEM_768`: two key exchanges in one handshake, that's hybrid
- the IKE_INTERMEDIATE exchange, a third round trip that carries the ML-KEM share
- `splitting IKE message (1249 bytes) into 2 fragments`
- on the wire: **1252 B + 70 B** back to back, then the ciphertext in one 1157 B packet

Measured on 2026-10-22: exactly those numbers. The sample output in
`learn/ipsec/key-exchange/README.md` shows the same run.

**Beat 1: bring up the hybrid tunnel (watch the live IKE exchange)**

In [ ]:
# Initiator: the swanctl.conf knob that adds ML-KEM on top of the classical DH group
#   proposals ending in '-ke1_mlkem768' is what makes this hybrid (the commented-out
#   line below it is the classical-only alternative); fragmentation is required
#   because the ML-KEM public key is too big to fit in one unfragmented IKE message.
initiator("grep -E -A1 'proposals|fragmentation' /usr/local/etc/swanctl/swanctl.conf");

In [ ]:
# Initiator: make sure no previous tunnel/capture is still running
initiator("swanctl --terminate --ike pqc-tunnel 2>/dev/null || true; true")
initiator("pkill -f [t]cpdump 2>/dev/null; true");

In [ ]:
# Initiator: capture IKE (UDP 500/4500), then initiate the hybrid child SA
initiator("tcpdump -i any --immediate-mode -U -n 'port 500 or port 4500' -w /tmp/hybrid.pcap &")
initiator("swanctl --initiate --child pqc-child");

In [ ]:
# Initiator: read the negotiated IKE SA suite off swanctl
suite = initiator("swanctl --list-sas | grep -iE 'ML_KEM|CURVE_25519' || true")
initiator("pkill -f [t]cpdump 2>/dev/null; sleep 1; true");

In [ ]:
# The ML-KEM pubkey always splits into exactly 2 fragments (Beat 2 measures the
# exact sizes); draw both arrows now, bolded, but no sizes yet: that's Beat 2's job
ipsec.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM pubkey)", ["", ""])
panel(ipsec.stage, step="IKE_AUTH  (confirm)", focus="IKE_INTERMEDIATE  (ML-KEM pubkey)", status=("ok", "ok"),
      rows=[("IKE SA suite", "CURVE_25519/KE1_ML_KEM_768", suite.text or "(no SA line)",
             "ML_KEM" in suite.text.upper())],
      notes=[("Round trips", "3 (adds the IKE_INTERMEDIATE exchange)")],
      banner="IKEv2 carries ML-KEM in an extra IKE_INTERMEDIATE round trip (RFC 9370) that classical IKEv2 does not have.")

**Beat 2: the fragmented ML-KEM key share on the wire**

In [ ]:
# Initiator: dump the raw capture; the fragmented IKE_INTERMEDIATE request is
# the biggest burst of back-to-back outbound packets (before the responder replies)
raw = initiator("tcpdump -r /tmp/hybrid.pcap -n -q 2>/dev/null")

In [ ]:
# Group the outbound packet sizes into bursts and keep the biggest one; that
# burst is the fragmented message; each element is one fragment's own size
bursts, cur = [], []
for line in raw.text.splitlines():
    if ' Out ' in line:
        cur.append(int(line.split()[-1]))
    elif ' In ' in line and cur:
        bursts.append(cur); cur = []
if cur:
    bursts.append(cur)
frags = [str(n) for n in max(bursts, key=sum)] if bursts else ['?']
total = str(sum(int(n) for n in frags))
print(f'fragments: {frags} (total {total} B)')

In [ ]:
# Draw one arrow per fragment actually seen on the wire, each tagged with its own size,
# this is what differentiates this panel from Beat 1's (same 2 arrows, now with real sizes)
last_step = ipsec.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM pubkey)", [f"{n} B" for n in frags])
panel(ipsec.stage, step=last_step, focus="IKE_INTERMEDIATE  (ML-KEM pubkey)", status=("ok", "ok"),
      rows=[("IKE_INTERMEDIATE message, before fragmentation", ">1280 B (~1322)", total + " B",
             total.isdigit() and int(total) > 1280),
            (f"Fragment 1/{len(frags)} on the wire", "~1252 B", frags[0] + " B",
             frags[0].isdigit() and abs(int(frags[0]) - 1252) <= 40),
            (f"Fragment {len(frags)}/{len(frags)} on the wire", "~70 B", frags[-1] + " B",
             frags[-1].isdigit() and abs(int(frags[-1]) - 70) <= 40)],
      banner=f"The ML-KEM public key makes the IKE_INTERMEDIATE message ~1322 B, past the ~1280 B IKE ceiling, so strongSwan splits it into {len(frags)} fragments.",
      banner_color=SLATE)

In [ ]:
# reset (terminate the tunnel, stop the capture)
initiator("swanctl --terminate --ike pqc-tunnel 2>/dev/null || true; true")
initiator("pkill -f [t]cpdump 2>/dev/null; true");

### Demo 2: post-quantum authentication

The key exchange is post-quantum now; the signatures aren't. What happens when they
are? Look for:

- ECDSA: `IKE_AUTH` fits in **one 958 B packet**
- `authentication of 'responder.pqc.lab' with ML_DSA_44 successful`
- `splitting IKE message (~6,900 bytes) into 6 fragments` (6,884 B on 2026-10-22,
  5 x 1248 + 953; strongSwan master is built from source, so expect a few bytes either way)
- the ML-KEM split doesn't change: 1249 B in 2 fragments, both times

**Heads up:** the strongSwan master build is **experimental**. Act 2 does the same
thing on shipping router code. `learn/ipsec/authentication/README.md` has sample
output.

**Beat 1: mutually authenticate with ECDSA certs**

In [ ]:
# host: mint ECDSA credentials for the two strongSwan peers (stack is
#       already up from preflight_up())
host("cd learn/ipsec/authentication && docker compose run --rm certgen ecdsa");

In [ ]:
# both peers: load the freshly minted credentials
auth_responder("swanctl --load-all")
auth_initiator("swanctl --load-all");

In [ ]:
# Initiator: bring up the tunnel; the log line names the authentication method
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true")
res = auth_initiator("swanctl --initiate --child auth-child")
suite = auth_initiator("swanctl --list-sas | grep -iE 'ML_KEM|CURVE_25519' || true")

In [ ]:
# Tag the IKE_AUTH arrows with what actually just ran, and bold them,
# this beat is about the authentication method, not the key exchange above it
ipsec_auth.stage.retag("IKE_AUTH  (cert + signature)", "ECDSA", direction="r")
panel(ipsec_auth.stage, step=5, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("Authentication method", "ECDSA_WITH_SHA256_DER",
             (grep(res.text, 'authentication of') or ['?'])[-1], 'ecdsa_with_sha256' in res.text.lower()),
            ("Key-exchange suite", "CURVE_25519/KE1_ML_KEM_768", suite.text, "ML_KEM" in suite.text.upper())],
      banner="Today's posture (e.g. 5G fronthaul): post-quantum ML-KEM key exchange with classical ECDSA authentication.",
      banner_color=BLUE)

In [ ]:
# reset the ECDSA tunnel before the ML-DSA rebuild
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true");

**Beat 2 (bleeding edge): ML-DSA-44 certificate authentication**

Switches the peers to the experimental strongSwan `master` image (already
compiled by `preflight_build()`) and reissues ML-DSA-44 certs.

In [ ]:
# host: switch to the experimental strongSwan master image, reissue ML-DSA-44 certs
host("cd learn/ipsec/authentication && docker compose -f docker-compose.yml -f docker-compose.mldsa.yml up -d")
host("cd learn/ipsec/authentication && docker compose -f docker-compose.yml -f docker-compose.mldsa.yml run --rm certgen ml-dsa-44");

In [ ]:
# both peers: load the ML-DSA credentials, then initiate
auth_responder("swanctl --load-all")
auth_initiator("swanctl --load-all")
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true")
res = auth_initiator("swanctl --initiate --child auth-child")

In [ ]:
# Retag the same arrows for this beat's actual auth method
ipsec_auth.stage.retag("IKE_AUTH  (cert + signature)", "ML-DSA-44", direction="r")
panel(ipsec_auth.stage, step=5, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("Authentication method", "ML_DSA_44",
             (grep(res.text, 'authentication of') or ['?'])[-1], 'ml_dsa_44' in res.text.lower())],
      banner="Frontier: both halves post-quantum (ML-KEM + ML-DSA). Experimental strongSwan master build; wire format still an IETF draft.",
      banner_color=MAGENTA)

**Beat 3: watch the ML-DSA cert + signature fragment IKE_AUTH**

In [ ]:
# host: search the initiator's log for IKE fragmentation lines, then pick out
# the biggest one; that's always the IKE_AUTH split (ML-KEM's is always ~1.2 KB)
frag = host("docker logs ike-auth-initiator 2>&1 | grep -i 'splitting IKE message'")
matches = re.findall(r'\((\d+) bytes\) into (\d+) fragments', frag.text)
auth_bytes, auth_frags = max(matches, key=lambda m: int(m[0])) if matches else ("6904", "6")
auth_frags = int(auth_frags)

In [ ]:
# Draw one arrow per real fragment: the fixed 2-way ML-KEM split (every run),
# plus one arrow per IKE_AUTH fragment, each tagged with its share of the real total
ipsec_auth.stage.set_fragments("IKE_INTERMEDIATE  (ML-KEM key share)", ["1252 B", "70 B"])
avg = int(auth_bytes) // auth_frags
last_step = ipsec_auth.stage.set_fragments("IKE_AUTH  (cert + signature)", [f"~{avg} B avg"] * auth_frags)

In [ ]:
panel(ipsec_auth.stage, step=last_step, focus="IKE_AUTH  (cert + signature)", status=("ok", "ok"),
      rows=[("IKE_INTERMEDIATE split (ML-KEM, every run)", "1249 B -> 2 fragments", "1249 B -> 2 fragments", True),
            ("IKE_AUTH split (ML-DSA cert + signature)", ">1280 B -> several fragments",
             f"{auth_bytes} B -> {auth_frags} fragments", int(auth_bytes) > 1280 and auth_frags > 2)],
      banner=f"The ML-DSA leaf cert + signature blow IKE_AUTH up to ~{auth_bytes} B -> {auth_frags} fragments; the ~1.2 KB ML-KEM split happens either way.",
      banner_color=SLATE)

In [ ]:
# reset (terminate the ML-DSA tunnel)
auth_initiator("swanctl --terminate --ike auth-tunnel 2>/dev/null || true; true");

That's where the bytes go: the key exchange adds about 1.2 KB, and ML-DSA
authentication adds about 6 KB more.

Something stuck? This clears it without taking the stacks down:

In [ ]:
# host: terminate both demo tunnels and stray captures, keep the stacks up
labs.panic()

---
<a id="act2-setup"></a>
## Starting at Act 2? Set up the routers

Skipping the containers (no Docker, or routers only)? Start here. This cell loads
the router and playbook helpers itself, so you don't need the Setup cell at the
top (Acts 2 and 3 only use those helpers, never the Act 1 ones). Then it
configures everything Acts 2 and 3 need, from whatever state the lab is in:

1. checks the [underlay](#underlay) (it never builds it)
2. tears down Act 3 objects, if a previous run left them (about 3 minutes)
3. on R1 and R3: `service internal`, the `TP-MLDSA65` trustpoint (mints and imports
   a fresh ML-DSA-65 PKI if either end is missing it), Demos 4, 6 and 7 undone, and
   Exercise 1 from `ipsec.md` with your [underlay](#underlay); if a trustpoint's
   certificate is for another address, it re-mints that too
4. NETCONF on all three, for Act 3
5. the same read-only check as the pre-flight

**This cell configures the routers**, unlike every other router cell here. On a
lab that's already in shape it only reads (about a minute). After a full pass
through Act 3 it takes 3 to 4 minutes.

Why does it backdate the certificates? So a router whose clock runs behind your
laptop doesn't see them as *not yet valid* (see
[ML-DSA trustpoints by hand](README.md#ml-dsa-trustpoints-by-hand)). The new PKI
starts a day ago.

In [ ]:
# R1, R2, R3: CONFIGURES the Act 2 starting state, then checks it
# (loads the helpers first, so this works as the first cell you run)
import sys, pathlib
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'ipsec_demo'):
    sys.path.insert(0, str(_demos / _p))
from ipsec_lab import *  # noqa: F401,F403  routers, playbooks, labs

setup()

---
<a id="act2"></a>
## Act 2: leave the laptop behind

**You type every command in this act at the router CLI**, one SSH session per
router. The cells here hold the commands, what to look for, and a read-only check
per demo that confirms you got the result.

### Demo 3: the classical baseline

**R1:**

```
show crypto ikev2 sa
ping 192.168.100.2
```

Read the algorithm line: `DH Grp:20, Auth sign: PSK, Auth verify: PSK`, `READY`,
5/5 pings. Then notice what's *missing*: no PQC line. Everything here falls to a
large enough quantum computer.

In [ ]:
# R1: check the baseline is classical (read-only)
router_check('Demo 3: classical baseline', 'R1, show crypto ikev2 sa', [
    Check('r1', 'show crypto ikev2 sa', 'SA state', 'READY', r'READY'),
    Check('r1', 'show crypto ikev2 sa', 'algorithms', 'DH Grp:20, Auth sign: PSK', r'DH Grp:20.*Auth sign: PSK'),
    Check('r1', 'show crypto ikev2 sa detailed', 'nothing post-quantum yet', 'absent', r'PQC|Quantum', present=False),
])

### Demo 4: the one-line upgrade

**R1, then the same on R3:**

```
conf t
 crypto ikev2 fragmentation mtu 1400
 crypto ikev2 proposal CLASSICAL-PROPOSAL
  pqc mlkem768 optional
end
```

Why fragmentation? The ML-KEM share pushes IKE messages past a typical path MTU,
and IKEv2 fragmentation (RFC 7383) splits them before IP has to. Why `optional`?
A peer that doesn't speak ML-KEM still gets a classical tunnel instead of none.

**Then on R1:** `clear crypto ikev2 sa`, `ping 192.168.100.2 repeat 20`,
`show crypto ikev2 sa detailed`. Why `repeat 20`? After the clear, IOS waited
between 4 and 24 seconds before it re-initiated in our runs, so a plain 5-ping
run can end 0/5 with no SA to show. Expect some dots first.

Look for `PQC Key Exchange: ML-KEM-768`, `Quantum-safe Encryption using PQC`,
`DH Grp:20` still there (that's the hybrid), and `MTU in use: 1372 bytes`.
`deploy/c8000/ipsec.md`, Exercise 3, has the same output.

In [ ]:
# R1, R3: check ML-KEM is negotiated and the config is on both ends (read-only)
router_check('Demo 4: one-line ML-KEM', 'R1 SA, R1 and R3 config', [
    Check('r1', 'show crypto ikev2 sa detailed', 'PQC key exchange', 'PQC Key Exchange: ML-KEM-768', r'PQC Key Exchange: ML-KEM-768'),
    Check('r1', 'show crypto ikev2 sa detailed', 'quantum-safe', 'Quantum-safe Encryption using PQC: ML-KEM-768', r'Quantum-safe Encryption using PQC: ML-KEM-768'),
    Check('r1', 'show crypto ikev2 sa detailed', 'DH group kept (hybrid)', 'DH Grp:20', r'DH Grp:20'),
    Check('r1', 'show crypto ikev2 sa detailed', 'fragmentation MTU', '1372 bytes', r'MTU in use: 1372'),
    Check('r1', 'show running-config | include pqc mlkem', 'config on R1', 'pqc mlkem768 optional', r'pqc mlkem768 optional'),
    Check('r3', 'show running-config | include pqc mlkem', 'config on R3', 'pqc mlkem768 optional', r'pqc mlkem768 optional'),
])

### Demo 5: weigh it on the wire

How many bytes did that one line add? R2 sits in the path, so capture there with
Embedded Packet Capture (EPC), on R2's interface toward R1. That's `Vlan12` below;
use yours from `underlay()` if it's different.

**R2:**

```
monitor capture CAP interface Vlan12 both
monitor capture CAP match ipv4 protocol udp any any
monitor capture CAP buffer size 20
monitor capture CAP start
```

**R1:** `clear crypto ikev2 sa`, `ping 192.168.100.2 repeat 20`. Don't stop
the capture until you see `!`: the new handshake showed up 12 to 24 s after the
clear in our runs (the handshake itself takes about 23 ms).

**R2, stop first, then read:**

```
monitor capture CAP stop
show monitor capture CAP buffer detailed
no monitor capture CAP
```

How do you tell the exchanges apart? Byte `0x3C` of the frame dump (row `0030`,
13th byte) is the IKE exchange type. That's 14 + 20 + 8 bytes of Ethernet, IP
and UDP, plus 18 into the IKE header. **34** is IKE_SA_INIT, **43** is
IKE_INTERMEDIATE (that's ML-KEM), **35** is IKE_AUTH. **37** is INFORMATIONAL:
the first two frames are the delete your clear sent, anything later is a
keepalive.

Measured on IOS XE 26.2, three runs, identical every time:

| Exchange | Request | Response | Total |
|---|---|---|---|
| IKE_SA_INIT (34) | 488 B | 555 B | 1,043 B |
| IKE_INTERMEDIATE (43) | 1,322 B | 1,226 B | **2,548 B** |
| IKE_AUTH (35), PSK | 682 B | 346 B | 1,028 B |

The 2,548 B matches `ipsec.md`, Exercise 6, to the byte. No check cell here: the
capture buffer is the result.

### Demo 6: an ML-DSA key, minted on the box

Can the router make an ML-DSA key itself? **R1:**

```
conf t
 crypto key generate ?          ! ec, rsa only
end
crypto key generate ?           ! ec, mldsa, rsa
crypto key generate mldsa param 65 label PQC-DEMO
crypto key zeroize mldsa PQC-DEMO
```

Notice that `mldsa` only shows up in exec mode, not under `conf t`. Expect
`% Generating MLDSA-65 keys, keys will be non-exportable...[OK] (elapsed time was
0 seconds)`. Answer `yes` if zeroize asks to confirm.

In [ ]:
# R1: the demo key is gone again (read-only)
router_check('Demo 6: demo key cleaned up', 'R1, show crypto key mypubkey all', [
    Check('r1', 'show crypto key mypubkey all | include Key name', 'PQC-DEMO zeroized', 'absent', r'PQC-DEMO', present=False),
    Check('r1', 'show crypto key mypubkey all | include Key name', 'trustpoint key untouched', 'TP-MLDSA65', r'TP-MLDSA65'),
])

### Demo 7: both pillars, on one SA

ML-KEM protects the key exchange. Now swap the PSK for ML-DSA certificates, so
authentication is post-quantum too.

The paste comes from `ipsec.md`, Exercise 5, "Swap PSK for ML-DSA". Its identity
lines carry each router's own underlay address and its peer's, so this cell prints
it for your lab:

In [ ]:
# laptop: Demo 7's paste for R1 and R3, with your underlay addresses
mldsa_snippets()

**Paste the R1 block on R1, then the R3 block on R3.** Expect the "exetremely
large" (sic) fragmentation warning, and possibly `%Shutting down Tunnel0`, on one
router or both; the paste already ends in `no shutdown`.

**R1:** `ping 192.168.100.2 repeat 20`, `show crypto ikev2 sa detailed`,
`show crypto session detail`. For a few seconds the old PSK SA sits *above* the
new one, in `DELETE` (`show crypto session detail` lists it as `Inactive`,
`Capabilities:FUQ`). The one you want is `READY` / `Active`.

Look for `Auth sign: MLDSA, Auth verify: MLDSA`, then `PQC Key Exchange:
ML-KEM-768`, then `Capabilities:DFUQ` (the legend at the top of the output says
what each letter means). `ipsec.md`,
Exercise 5, "Verify", has the same output.

In [ ]:
# R1, R3: both pillars on one SA (read-only)
router_check('Demo 7: ML-KEM + ML-DSA', 'R1 and R3', [
    Check('r1', 'show crypto ikev2 sa detailed', 'authentication', 'Auth sign: MLDSA, Auth verify: MLDSA', r'Auth sign: MLDSA, Auth verify: MLDSA'),
    Check('r1', 'show crypto ikev2 sa detailed', 'key exchange', 'PQC Key Exchange: ML-KEM-768', r'PQC Key Exchange: ML-KEM-768'),
    Check('r1', 'show crypto session detail', 'capabilities', 'Capabilities:DFUQ', r'Capabilities:\S*Q'),
    Check('r3', 'show crypto ikev2 sa detailed', 'authentication', 'Auth sign: MLDSA, Auth verify: MLDSA', r'Auth sign: MLDSA, Auth verify: MLDSA'),
])

---
<a id="act3"></a>
## Act 3: now do it to the fabric

Typing it on two routers is fine. What about fifty? Act 3 builds a classical hub
and spoke (R2 the hub, R1 and R3 the spokes) with Ansible over NETCONF, then adds
ML-KEM to all of it with one playbook.

### Get the routers ready

`ipsec-baseline.yml` deletes the hand-built `CLASSICAL-*` objects, and it can't
remove `CLASSICAL-TS` while Demo 7's `MLDSA-IPSEC` still uses it. So first,
**paste this on R1 and R3** to release Tunnel0:

```
interface Tunnel0
 shutdown
 no tunnel protection ipsec profile MLDSA-IPSEC
exit
no crypto ipsec profile MLDSA-IPSEC
no crypto ikev2 profile MLDSA-PROFILE
```

Skip it and the baseline fails partway through, on the transform-set removal. Run
the check below (it takes a few seconds), then start the baseline.

In [ ]:
# R1, R3: Tunnel0 is released, so the baseline can clean up (read-only)
router_check('Act 3 reset done', 'R1 and R3', [
    Check(h, 'show running-config | include MLDSA-IPSEC|MLDSA-PROFILE', 'ML-DSA profile removed', 'absent', r'MLDSA-', present=False)
    for h in ('r1', 'r3')
])

In [ ]:
# Ansible: classical hub and spoke, in the background
baseline = playbook_bg('ansible-playbook ipsec-baseline.yml')

It runs in the background, so you can read ahead. Collect it with the next cell;
if it's still running, that cell waits and says so.

In [ ]:
# Ansible: wait for the baseline and show how it went
playbook_wait(baseline);

### Demo 8: Ansible pushes ML-KEM to the fabric

Run it twice. The first run changes something on every router; the second run
finds nothing to do. That second run is the point: the playbook is idempotent,
so it's safe to run on a schedule.

Captured run: **39 s** with `changed=2` on each router, then **31 s** with
`changed=0` (`captured/03-ipsec-pq-mlkem/`).

Why can the panel say 1 when the recap says 2? Every playbook ends with `verify.yml`, whose posture report is forced to `changed` so it stays on screen. The panel subtracts it and counts config changes only, which is what the captured runs show.

These cells keep the `PLAY RECAP` on screen. **Heads up:** in a plain terminal the repo's `pqc_lab` callback hides the recap when a run succeeds. Run them from here, or prefix `ANSIBLE_STDOUT_CALLBACK=default` in a terminal (plus `-e @../../../demos/common/lab.yml`; see [Running a playbook from a terminal](../README.md#run-a-playbook-from-a-terminal)).

In [ ]:
# Ansible: add ML-KEM to the proposal on all three routers
playbook('ansible-playbook ipsec-pq-mlkem.yml', expect_changed=2);

In [ ]:
# Ansible: same playbook again; nothing to do is the point
playbook('ansible-playbook ipsec-pq-mlkem.yml', expect_changed=0);

In [ ]:
# R2: the hub negotiated ML-KEM with both spokes
r2('show crypto ikev2 sa detailed | include PQC');

Leave ML-KEM in place for now: Demo 9 needs it.

### Demo 9: the guardrail refuses

ML-KEM and a post-quantum PPK are two alternative paths to a quantum-safe key
exchange, and the roles never stack them on the same box. So what happens if you
ask for PPK while ML-KEM is configured? The playbook checks first, and refuses.

Expected: fails in seconds, `changed=0`, says ML-KEM is still configured on
r1, r2 and r3, names `ansible-playbook ipsec-pq-mlkem.yml -e state=absent`,
and ends with "Nothing has been changed on any router."

That wording comes from `roles/common/tasks/refuse-stacked-pq-path.yml`. This
direction isn't in `captured/` yet; the mirror image is, in
`captured/02-ipsec-pq-ppk/negative-test-mlkem-refuses-while-ppk-present.log`
(worded by an older version of the guard).

In [ ]:
# Ansible: ask for PPK while ML-KEM is configured; it should refuse
playbook('ansible-playbook ipsec-pq-ppk.yml', expect_refusal=True, tail=25);

---
<a id="after"></a>
## Clean up

Take ML-KEM off (about 65 s in the captured run). The baseline under it stays up.

In [ ]:
# Ansible: remove the ML-KEM line from LAB-PROPOSAL on all three routers
playbook('ansible-playbook ipsec-pq-mlkem.yml -e state=absent');

Want the routers bare, with no automation objects at all? Remove the baseline too:

In [ ]:
# Ansible: remove the hub and spoke (Tunnel101/102 and every PQAUTO- object)
playbook('ansible-playbook ipsec-baseline.yml -e state=absent');

On the routers: `no monitor capture CAP` on R2 if it's still defined, then
`write memory` everywhere. Keep `TP-MLDSA65` for the next run, or remove it with
`no crypto pki trustpoint TP-MLDSA65`.

On the laptop: containers down, creds removed, `learn/ipsec` back to the
committed state.

In [ ]:
# host: Act 1 stacks down, generated keys and certs removed, learn/ipsec back to the committed state
labs.cleanup()

---
<a id="again"></a>
## Run it again

The baseline playbook deleted the hand-built tunnel, so going again means:

1. Run both teardown cells in [Clean up](#after).
2. Run [`setup()`](#act2-setup), or re-paste Exercise 1 from
   `deploy/c8000/ipsec.md` on R1 and R3 by hand.
3. `labs.cleanup()`, then `labs.up()`, so Demo 2 starts from the default
   images again rather than the ML-DSA ones it left running.
4. Rerun the [router pre-flight](#preflight) until every row is green.